<a href="https://colab.research.google.com/github/pikey-msc/RiesgosFinancieros/blob/master/2026-1/CVA_Derivados_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Informe de Investigación: CVA y Derivados – Medidas de Exposición y Cálculo Avanzado

## 1. Introducción: El Cambio de Paradigma en la Valoración de Derivados

La gestión de riesgos financieros, y en particular la valoración de instrumentos derivados, sufrió una transformación tectónica tras la crisis financiera global de 2008. Anteriormente, el marco estándar de valoración asumía que las contrapartes principales —grandes bancos de inversión y corporaciones multinacionales— poseían una calidad crediticia tal que el riesgo de incumplimiento podía considerarse despreciable para efectos de *pricing* diario. La quiebra de Lehman Brothers en septiembre de 2008 demolió esta suposición, demostrando que el riesgo de crédito de contraparte (CCR, por sus siglas en inglés) es un componente intrínseco y material en la valoración de cualquier contrato bilateral no compensado centralmente.

Este informe técnico, diseñado para un nivel avanzado en la formación actuarial, disecciona el Ajuste de Valoración por Crédito (CVA, *Credit Valuation Adjustment*), que representa el precio de mercado del riesgo de crédito de la contraparte. Desde una perspectiva actuarial, el CVA no es simplemente un ajuste contable; es el valor esperado actuarial de las pérdidas futuras descontadas bajo la medida de probabilidad neutral al riesgo. La comprensión profunda del CVA requiere una convergencia de disciplinas: el cálculo estocástico para modelar la evolución de los factores de riesgo de mercado, la teoría de la medida para definir rigurosamente las métricas de exposición, y la simulación numérica (Monte Carlo) para la cuantificación efectiva.

En las secciones subsiguientes, transitaremos desde los fundamentos teóricos hacia la implementación práctica en Python, proporcionando una guía exhaustiva que permite al actuario no solo calcular un número, sino comprender la dinámica estocástica subyacente que lo genera.

### 1.1 Naturaleza Bilateral del Riesgo y Definición Económica

A diferencia del riesgo de crédito tradicional en un préstamo bancario, donde la exposición es unilateral (el banco presta, el cliente debe; la exposición del banco es el nocional pendiente), el riesgo en derivados es **bilateral**. El valor de mercado (*Mark-to-Market*, MtM) de un derivado, como un Swap de Tasas de Interés (IRS) o un Forward de Divisas, fluctúa estocásticamente pudiendo ser positivo o negativo para cualquiera de las dos partes durante la vida del contrato.

*   **Si el MtM es positivo para la institución:** La institución tiene un activo (una cuenta por cobrar teórica). Si la contraparte incumple en este momento, la institución sufre una pérdida equivalente al costo de reemplazo del contrato.
*   **Si el MtM es negativo para la institución:** La institución tiene un pasivo. Si la contraparte incumple, la institución teóricamente "gana" al no tener que pagar el valor total (sujeto a las leyes de quiebra y cláusulas de *walk-away*, aunque esto introduce el concepto de DVA o *Debt Valuation Adjustment*).

Económicamente, el CVA se define como la diferencia entre el valor del portafolio libre de riesgo y el valor del portafolio considerando el riesgo de incumplimiento:

$$\text{Valor}_{\text{Riesgoso}} = \text{Valor}_{\text{Libre de Riesgo}} - \text{CVA}$$

Esta ecuación fundamental implica que el CVA es siempre un costo (un valor negativo que reduce el activo) desde la perspectiva de quien posee la exposición positiva. Para el actuario, el CVA es análogo a una prima de seguro compleja: es el costo de asegurar el flujo de caja del derivado contra el evento de default de la contraparte, donde la "suma asegurada" es variable y estocástica (la exposición futura).




## 2. Fundamentos Teóricos: Medidas de Exposición bajo Incertidumbre

Para calcular el CVA, primero debemos cuantificar cuánto dinero está en riesgo en cualquier momento futuro $t$. Dado que los mercados son inciertos, la exposición en $t$ es una variable aleatoria $E(t)$. El análisis riguroso de esta variable requiere definiciones precisas basadas en la teoría de la probabilidad y la medida.

### 2.1 Definiciones en el Espacio de Probabilidad

Consideremos un espacio de probabilidad filtrado $(\Omega, \mathcal{F}, (\mathcal{F}_t)_{t \ge 0}, \mathbb{Q})$, donde $\Omega$ representa el conjunto de todos los posibles estados del mundo, $\mathcal{F}$ es la $\sigma$-álgebra que contiene todos los eventos posibles, y $\mathbb{Q}$ es la medida de probabilidad neutral al riesgo, bajo la cual los precios de los activos descontados son martingalas.

Sea $V(t, \omega)$ el valor de mercado de un portafolio de derivados en el tiempo $t$ bajo el escenario de mercado $\omega \in \Omega$. La **Exposición** (*Exposure*) se define como la parte positiva de este valor, ya que legalmente no se puede perder más de lo que se tiene derecho a recibir (ignorando por el momento el riesgo de liquidación o *settlement risk*):

$$E(t, \omega) = \max(V(t, \omega), 0) = [V(t, \omega)]^+$$

Esta nolinealidad (la función máximo) es crucial. Transforma distribuciones que podrían ser simétricas (como un Forward con valor inicial cero) en distribuciones sesgadas con masa en cero. Esto implica que, incluso si el valor esperado del derivado es cero ($E[V(t)] = 0$), la Exposición Esperada es estrictamente positiva ($E[E(t)] > 0$).

### 2.2 Métricas de Exposición (Exposure Metrics)

Las instituciones financieras y los reguladores (Basilea III/IV) utilizan una batería de métricas derivadas de la distribución de $E(t)$ para gestionar diferentes aspectos del riesgo. Es vital diferenciar entre métricas de "tendencia central" (usadas para *pricing* y CVA) y métricas de "cola" (usadas para capital económico y límites de crédito).

La siguiente tabla resume las métricas clave y sus aplicaciones:

| Métrica | Definición Matemática | Interpretación Actuarial/Financiera | Uso Principal |
| :--- | :--- | :--- | :--- |
| **Exposición Esperada (EE)** | $EE(t) = \mathbb{E}^\mathbb{Q}[E(t)]$ | El valor medio de la pérdida potencial en $t$ si ocurre el default. | Cálculo de CVA, Reservas. |
| **Exposición Futura Potencial (PFE)** | $PFE_\alpha(t) = \inf \{x : \mathbb{P}(E(t) \le x) \ge \alpha\}$ | El "peor caso" de exposición con un nivel de confianza $\alpha$ (ej. 99%). Análogo al VaR. | Límites de crédito, Capital Económico. |
| **Exposición Positiva Esperada (EPE)** | $EPE = \frac{1}{T} \int_0^T EE(t) dt$ | El promedio temporal de la exposición esperada. | Simplificación para cargas de capital (Basilea). |
| **Exposición Máxima (Peak Exposure)** | $\max_{t \in} PFE_\alpha(t)$ | El punto más alto de la curva de PFE durante la vida del contrato. | Dimensionamiento de líneas de crédito. |

#### Profundización en la Exposición Esperada (EE)
La Exposición Esperada es la función $t \mapsto EE(t)$. Esta función no es constante. Para un Swap de Tasas de Interés (IRS), típicamente tiene forma de campana: comienza en cero, sube debido a la difusión de la volatilidad de las tasas, y luego baja hacia el vencimiento debido al efecto de "pull-to-par" (menos flujos de caja restantes). Para una opción, la EE puede ser monótona creciente. El actuario debe modelar la curva completa $EE(t)$ para integrar el CVA correctamente.

#### Profundización en la Exposición Futura Potencial (PFE)
La PFE responde a la pregunta: "¿Cuánto podría llegar a deberme la contraparte en un escenario extremo?". Si la EE es el promedio, la PFE es la cola de la distribución. Matemáticamente, es el cuantil $\alpha$ de la distribución de $[V(t)]^+$. Es fundamental entender que la PFE suele ser significativamente mayor que la EE. Un error común es subestimar el capital necesario basándose solo en promedios (EE) en lugar de colas (PFE).

### 2.3 Mitigantes de Riesgo: Netting y Colateral

La exposición "bruta" descrita anteriormente raramente es la exposición real debido a los acuerdos legales de mitigación de riesgo, principalmente el **ISDA Master Agreement** y el **Credit Support Annex (CSA)**.

#### Efecto del Netting (Compensación)
El netting permite que, en caso de default, los valores positivos y negativos de diferentes operaciones con la misma contraparte se compensen. Matemáticamente, esto cambia el orden de la suma y la función máximo.
Sin Netting (Suma de Exposiciones):
$$E_{\text{Sin Netting}}(t) = \sum_{i=1}^N \max(V_i(t), 0)$$Con Netting (Exposición de la Suma):$$E_{\text{Netting Set}}(t) = \max \left( \sum_{i=1}^N V_i(t), 0 \right)$$
Debido a la desigualdad triangular y la subaditividad de la función máximo, $E_{\text{Netting Set}}(t) \le E_{\text{Sin Netting}}(t)$. La reducción de exposición gracias al netting es masiva en portafolios grandes y diversificados.

#### Efecto del Colateral (Garantías)
Los acuerdos CSA estipulan que si la exposición supera un cierto umbral (*Threshold*), la contraparte debe depositar colateral (efectivo o bonos) para cubrir la diferencia. Sin embargo, el colateral no elimina el riesgo instantáneamente debido al **Periodo de Riesgo de Margen (MPOR)**. El MPOR es el tiempo que transcurre desde la última llamada de margen exitosa hasta que se liquidan las posiciones tras un default (típicamente 10 a 20 días). Durante este periodo, la exposición puede crecer sin colateral adicional.
$$E_{\text{Colateralizado}}(t) = \max(V(t) - C(t - \delta), 0)$$
Donde $C(t-\delta)$ es el colateral determinado en el tiempo $t-\delta$ (antes del periodo de riesgo). Modelar el MPOR requiere simular la evolución del valor del portafolio dentro de ese intervalo corto de tiempo con alta precisión.




## 3. Modelado Estocástico y Generación de Escenarios (Monte Carlo)

Dado que las fórmulas cerradas para la EE solo existen para casos triviales, el estándar de la industria es la Simulación de Monte Carlo. Este enfoque numérico permite valorar la opcionalidad compleja, el netting y las reglas de colateral dependientes de la trayectoria (*path-dependent*).

### 3.1 Arquitectura de Simulación

El proceso para calcular CVA mediante Monte Carlo consta de tres fases distintas:
1.  **Generación de Escenarios (Calibración y Evolución):** Simulación de los factores de riesgo de mercado (tasas de interés, tipos de cambio, precios de acciones) bajo la medida $\mathbb{Q}$ hasta el vencimiento más largo del portafolio.
2.  **Valoración (Pricing):** En cada nodo de tiempo $t_i$ y para cada trayectoria $j$, se calcula el valor de cada instrumento $k$ en el portafolio: $V_{ijk}$.
3.  **Agregación:** Se aplica netting y colateral a nivel de contraparte para obtener $E_{ij}$, y luego se promedia sobre $j$ para obtener $EE(t_i)$.

### 3.2 Dinámica de los Factores de Riesgo

Para un activo de renta variable o tipo de cambio, el modelo base es el Movimiento Browniano Geométrico (GBM), descrito por la Ecuación Diferencial Estocástica (SDE):

$$dS_t = (r_t - q_t) S_t dt + \sigma S_t dW_t$$

Donde $r_t$ es la tasa libre de riesgo, $q_t$ el rendimiento por dividendos, y $dW_t$ un proceso de Wiener estándar.

Para tasas de interés, el GBM no es adecuado (las tasas pueden ser negativas o revertir a la media). Se utilizan modelos de reversión a la media como **Hull-White (1 factor)**:

$$dr_t = [\theta(t) - a r_t] dt + \sigma dW_t$$

Donde $a$ es la velocidad de reversión a la media y $\theta(t)$ se calibra para ajustar la estructura temporal de tasas de interés actual. La simulación de tasas es crítica porque afecta tanto al descuento de flujos como a la valoración de instrumentos como Swaps.

### 3.3 Discretización y Vectorización

Para implementar esto computacionalmente, discretizamos la SDE usando el esquema de Euler-Maruyama o soluciones exactas. Para el GBM, la solución exacta vectorizada es:

$$ S(t_{i+1}) = S(t_i) \cdot \exp \left( (r - \frac{1}{2}\sigma^2)\Delta t + \sigma \sqrt{\Delta t} Z_{i+1} \right) $$

Donde $Z \sim N(0, 1)$. En Python, utilizando bibliotecas como `numpy`, evitamos bucles `for` sobre las trayectorias. Operamos con matrices de dimensión `(N_simulaciones, N_pasos)`, lo que permite aprovechar las instrucciones SIMD de los procesadores modernos y acelerar el cálculo por órdenes de magnitud.




## 4. Cálculo del CVA: Derivación Integral y Discretización

Una vez obtenidos los perfiles de exposición $EE(t)$, procedemos al cálculo del CVA propiamente dicho.

### 4.1 La Integral de CVA

El CVA es el valor esperado de la pérdida. Si definimos $\tau$ como el tiempo aleatorio de default de la contraparte y $LGD$ como la pérdida en caso de incumplimiento (típicamente $1 - \text{Tasa de Recuperación}$), el CVA es:

$$CVA = \mathbb{E}^\mathbb{Q} \left$$

Asumiendo independencia entre la exposición y el default (ausencia de *Wrong-Way Risk*), podemos separar la esperanza de la exposición y la probabilidad de default:

$$CVA = LGD \int_0^T \mathbb{E}^\mathbb{Q} \left dPD(0, t)$$

Aquí, $\mathbb{E}^\mathbb{Q}$ es la Exposición Esperada Descontada (*Discounted Expected Exposure*, $DiscEE(t)$). El término $dPD(0, t)$ es la densidad de probabilidad de default en el tiempo $t$.

### 4.2 Probabilidad de Default (PD) y Hazard Rates

La probabilidad de default se extrae de los precios de mercado de los *Credit Default Swaps* (CDS). Si un CDS a 5 años cotiza con un spread de $S_{cds}$ puntos básicos, esto implica una probabilidad de riesgo neutral de incumplimiento.
La relación básica, asumiendo una intensidad de default ($\lambda$) constante (modelo de Poisson), es:

$$PD(0, t) = 1 - e^{-\lambda t} \approx 1 - e^{-\frac{S_{cds}}{LGD} t}$$

Para una curva de crédito completa, se utiliza un proceso de *bootstrapping* para hallar las intensidades $\lambda_i$ por tramos que repliquen los precios de mercado de los CDS para diferentes vencimientos (1Y, 3Y, 5Y, etc.).

### 4.3 Aproximación Numérica (Discretización)

En la práctica, la integral continua se aproxima mediante una suma discreta sobre la rejilla de simulación $t_0, t_1,..., t_N$. Una aproximación robusta es la regla del punto medio o trapezoidal:

$$ CVA \approx LGD \sum_{i=1}^N \left( \frac{DiscEE(t_{i-1}) + DiscEE(t_i)}{2} \right) \cdot \left( PD(0, t_i) - PD(0, t_{i-1}) \right) $$

El término $(PD(0, t_i) - PD(0, t_{i-1}))$ representa la **probabilidad marginal de default** en el intervalo $[t_{i-1}, t_i]$. Es crucial notar que esta probabilidad marginal suele ser decreciente en el tiempo si la intensidad es constante (debido a que la probabilidad de supervivencia disminuye), pero puede tener formas complejas si la curva de crédito es invertida.




## 5. Implementación Computacional en Python

A continuación, presentamos una implementación completa y modular en Python. Este código está diseñado para ejecutarse en Google Colab y utiliza `numpy` para cálculos vectorizados de alto rendimiento y `matplotlib` para la visualización de los perfiles de exposición.

El ejemplo modela el CVA de un portafolio simple consistente en una **Opción Call Europea**. Aunque un portafolio real incluiría Swaps y Forwards, la opción Call es ideal para fines didácticos porque tiene una solución cerrada para su valoración (Black-Scholes), lo que nos permite centrarnos en la mecánica del CVA (simulación de escenarios y agregación) sin perdernos en la complejidad de la valoración de Swaps.

### Instrucciones para el Estudiante
Copie las siguientes celdas en un cuaderno de Google Colab o Jupyter. El código está comentado extensamente para explicar cada paso del algoritmo.

### Celda 1: Configuración del Entorno y Parámetros
Esta celda importa las bibliotecas necesarias y define los parámetros del mercado y del contrato. Note el uso de `seaborn` para mejorar la estética de los gráficos.



In [ ]:

# ==============================================================================
# BLOQUE 1: IMPORTACIÓN DE LIBRERÍAS Y DEFINICIÓN DE PARÁMETROS
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as si
import seaborn as sns

# Configuración visual para gráficos estilo reporte profesional
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

# --- 1.1 Parámetros del Mercado y del Contrato ---
# Características del Activo Subyacente
S0 = 100.0       # Precio inicial del activo (Spot)
r = 0.05         # Tasa libre de riesgo (5% anual, composición continua)
sigma = 0.25     # Volatilidad anual del activo (25%)
q = 0.0          # Dividendos (asumimos 0 para simplificar)

# Características del Instrumento (Opción Call Europea)
K = 100.0        # Precio de ejercicio (Strike)
T = 5.0          # Vencimiento del contrato (años)
Type = 'Call'    # Tipo de opción

# Características de la Contraparte (Riesgo de Crédito)
# Asumimos una intensidad de default (lambda) constante derivada de un CDS Spread
# Spread ~ lambda * (1 - R). Si Spread = 200bps y R = 40%, lambda ~ 0.02 / 0.6 = 0.0333
lambda_credit = 0.03  # Intensidad de default (Hazard Rate)
recovery_rate = 0.40  # Tasa de recuperación
LGD = 1.0 - recovery_rate # Loss Given Default (60%)

# --- 1.2 Configuración de la Simulación Monte Carlo ---
n_sims = 5000    # Número de escenarios (trayectorias)
n_steps = 100    # Número de pasos de tiempo (discretización temporal)
dt = T / n_steps # Tamaño del paso de tiempo
time_grid = np.linspace(0, T, n_steps + 1) # Vector de tiempos:

print(f"--- Configuración de Simulación ---")
print(f"Simulaciones: {n_sims}")
print(f"Pasos de tiempo: {n_steps} (dt = {dt:.4f} años)")
print(f"Instrumento: {Type} Option, Strike={K}, Vencimiento={T} años")
print(f"Parámetros de Crédito: LGD={LGD:.0%}, Intensidad={lambda_credit}")




### Celda 2: Motor de Simulación (Geometric Brownian Motion)
Aquí implementamos la generación de trayectorias. Utilizamos la solución exacta de la SDE del Movimiento Browniano Geométrico. La matriz `S` tendrá dimensiones `(n_sims, n_steps + 1)`.



In [ ]:

# ==============================================================================
# BLOQUE 2: MOTOR DE SIMULACIÓN DE ESCENARIOS (MONTE CARLO)
# ==============================================================================

def simulate_gbm_paths(S0, r, sigma, dt, n_sims, n_steps):

    ## Genera trayectorias de precios usando el Movimiento Browniano Geométrico.
    ## Retorna una matriz (n_sims, n_steps + 1).

    # Fijar semilla para reproducibilidad
    np.random.seed(42)

    # 1. Generar choques aleatorios Z ~ N(0, 1)
    # Dimensiones: (n_sims, n_steps)
    Z = np.random.standard_normal((n_sims, n_steps))

    # 2. Precomputar términos deterministas de la SDE
    # ln(S_t) = ln(S_0) + (r - 0.5*sigma^2)t + sigma*W_t
    drift = (r - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt)

    # 3. Evolución de precios
    # Inicializamos la matriz de precios con ceros
    S = np.zeros((n_sims, n_steps + 1))
    S[:, 0] = S0

    # Bucle eficiente sobre los pasos de tiempo (vectorizado en simulaciones)
    for t in range(1, n_steps + 1):
        # S(t) = S(t-1) * exp(drift + diffusion * Z)
        S[:, t] = S[:, t-1] * np.exp(drift + diffusion * Z[:, t-1])

    return S

# Ejecutar la simulación
S_paths = simulate_gbm_paths(S0, r, sigma, dt, n_sims, n_steps)

# Visualización de las primeras 50 trayectorias para validación visual
plt.figure()
plt.plot(time_grid, S_paths[:50, :].T, lw=1, alpha=0.4)
plt.title(f"Simulación de {n_sims} Trayectorias de Mercado (Activo Subyacente)")
plt.xlabel("Tiempo (Años)")
plt.ylabel("Precio del Activo ($)")
plt.show()




### Celda 3: Valoración en Nodos Futuros (Mark-to-Market Cube)
Esta es la fase crítica. Para calcular la exposición en $t=2$ años, necesitamos saber cuánto valdrá la opción en $t=2$ años para cada trayectoria simulada. Usamos la fórmula de Black-Scholes, pero cuidado: el tiempo hasta el vencimiento (`T - t`) disminuye a medida que avanzamos en la simulación. En $t=T$, el valor es simplemente el Payoff intrínseco $\max(S_T - K, 0)$.



In [ ]:

# ==============================================================================
# BLOQUE 3: VALORACIÓN FUTURA (MARK-TO-MARKET CUBE)
# ==============================================================================

def black_scholes_price(S, K, T_remaining, r, sigma, option_type='Call'):

    # Valoración vectorizada usando Black-Scholes para un tiempo remanente dado.

    # Si estamos en el vencimiento (o muy cerca), devolvemos el payoff intrínseco
    if T_remaining <= 1e-6:
        if option_type == 'Call':
            return np.maximum(S - K, 0)
        else:
            return np.maximum(K - S, 0)

    # Cálculo estándar de d1 y d2
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T_remaining) / (sigma * np.sqrt(T_remaining))
    d2 = d1 - sigma * np.sqrt(T_remaining)

    if option_type == 'Call':
        price = S * si.norm.cdf(d1) - K * np.exp(-r * T_remaining) * si.norm.cdf(d2)
    else:
        price = K * np.exp(-r * T_remaining) * si.norm.cdf(-d2) - S * si.norm.cdf(-d1)

    return price

# Inicializamos la matriz de MtM (Mark-to-Market)
MtM_cube = np.zeros_like(S_paths)

# Bucle sobre cada paso de tiempo en el grid
for i, t_current in enumerate(time_grid):
    # Tiempo restante hasta el vencimiento del contrato
    T_remaining = T - t_current

    # Valoramos la opción para TODOS los escenarios en este instante t
    # S_paths[:, i] es un vector de precios spot en el tiempo t
    MtM_cube[:, i] = black_scholes_price(S_paths[:, i], K, T_remaining, r, sigma, Type)

# Cálculo de la Exposición
# Exposure = max(MtM, 0).
# Para una opción comprada, el precio siempre es >= 0, así que Exposure == MtM.
# Si fuera un Swap, tendríamos valores negativos.
Exposure_cube = np.maximum(MtM_cube, 0)

print("Cubo de Valoración generado exitosamente.")
print(f"Dimensiones: {Exposure_cube.shape}")




### Celda 4: Cálculo de Perfiles de Exposición (EE, PFE)
Agregamos los resultados del cubo de exposición para obtener las métricas actuariales.



In [ ]:

# ==============================================================================
# BLOQUE 4: CÁLCULO DE PERFILES DE EXPOSICIÓN (EE, PFE)
# ==============================================================================

# 1. Expected Exposure (EE): Promedio a través de las simulaciones (eje 0)
EE = np.mean(Exposure_cube, axis=0)

# 2. Potential Future Exposure (PFE): Cuantiles (95% y 99%)
PFE_95 = np.percentile(Exposure_cube, 95, axis=0)
PFE_99 = np.percentile(Exposure_cube, 99, axis=0)

# 3. Discounted Expected Exposure (DiscEE)
# Necesario para el cálculo de CVA. Descontamos el EE al valor presente.
# Factor de descuento determinista (tasa r constante): exp(-r * t)
discount_factors = np.exp(-r * time_grid)
Disc_EE = EE * discount_factors

# --- Visualización de los Perfiles ---
plt.figure()

# PFE (Colas de la distribución)
plt.plot(time_grid, PFE_99, color='red', linestyle='--', label='PFE 99% (Límite de Crédito)')
plt.plot(time_grid, PFE_95, color='orange', linestyle='--', label='PFE 95%')

# EE (Tendencia central)
plt.plot(time_grid, EE, color='blue', linewidth=2, label='Expected Exposure (EE)')

# Rellenar área bajo la curva de EE para denotar la "masa" de exposición esperada
plt.fill_between(time_grid, 0, EE, color='blue', alpha=0.1)

plt.title(f"Perfiles de Exposición de Riesgo de Contraparte ({Type} Option)")
plt.xlabel("Tiempo (Años)")
plt.ylabel("Exposición ($)")
plt.legend()
plt.show()

# Insight: Note cómo el perfil comienza en el valor actual, sube por la difusión
# (incertidumbre) y finalmente converge al payoff en T.




### Celda 5: Integración Numérica del CVA
Finalmente, combinamos el perfil de Exposición Esperada Descontada (`Disc_EE`) con la curva de probabilidad de default.



In [ ]:

# ==============================================================================
# BLOQUE 5: CÁLCULO DEL CVA (INTEGRACIÓN NUMÉRICA)
# ==============================================================================

# 1. Calcular la Probabilidad de Default Acumulada (Cumulative PD)
# PD(t) = 1 - exp(-lambda * t)
PD_cum = 1 - np.exp(-lambda_credit * time_grid)

# 2. Calcular la Probabilidad Marginal de Default (dPD) en cada intervalo
# dPD[i] = PD(t_i) - PD(t_{i-1})
# Usamos np.diff, que retorna un array de longitud n_steps
marginal_PD = np.diff(PD_cum)

# 3. Preparar el Discounted EE para la integración
# Usamos la regla del punto medio para mayor precisión: (EE[i-1] + EE[i]) / 2
# Disc_EE tiene longitud n_steps + 1
Disc_EE_midpoint = 0.5 * (Disc_EE[:-1] + Disc_EE[1:])

# 4. Suma del CVA
# CVA = LGD * Sum( DiscEE_midpoint * marginal_PD )
CVA_value = LGD * np.sum(Disc_EE_midpoint * marginal_PD)

# --- Reporte de Resultados ---
risk_free_value = MtM_cube # Valor en t=0 (es igual para todas las simulaciones)
risky_value = risk_free_value - CVA_value

print(f"====== REPORTE DE VALORACIÓN ======")
print(f"Valor Libre de Riesgo (Risk-Free):   ${risk_free_value:,.2f}")
print(f"Ajuste por CVA (Costo de Crédito):   ${CVA_value:,.2f}")
print(f"Valor Riesgoso (Risky Value):        ${risky_value:,.2f}")
print(f"-----------------------------------")
print(f"Impacto del CVA en Valoración:       -{ (CVA_value / risk_free_value):.2%} ")

# Cálculo de spread equivalente (en puntos básicos)
# Si quisiéramos cobrar este CVA como un spread anual sobre el nocional (aprox)
# Spread ~ CVA / (Duración * Nocional) -> estimación rápida




## 6. Análisis Avanzado e Insights de Segundo Orden

La implementación anterior proporciona el valor numérico, pero la interpretación actuarial requiere ir más allá de la cifra bruta.

### 6.1 Wrong-Way Risk (WWR): La Correlación Letal
La fórmula empleada asume independencia entre la exposición y el default: $E[V \cdot \mathbb{I}_{def}] = E[V] \cdot E[\mathbb{I}_{def}]$. Esta asunción falla catastróficamente en presencia de **Wrong-Way Risk (WWR)**.

*   **WWR Específico:** Ocurre cuando la exposición futura está fuertemente correlacionada con la solvencia legal de la contraparte. Ejemplo clásico: Aceptar bonos de la propia contraparte como colateral, o comprar una opción Put sobre las acciones de la contraparte. Si la contraparte quiebra, su acción cae, la Put vale mucho (exposición alta), pero la contraparte no puede pagar.
*   **WWR General:** Ocurre por correlación macroeconómica. Ejemplo: Un banco chileno que tiene un Swap de Tasa Cruzada (CCS) con una corporación que recibe dólares y paga pesos. Si el peso se devalúa fuertemente, el banco gana en el derivado (exposición alta), pero la deuda en dólares de la corporación se vuelve impagable, aumentando su probabilidad de default.

**Impacto en Modelos:** Para capturar WWR, no se puede usar la fórmula de integral simple. Se requiere simular la intensidad de default $\lambda_t$ como un proceso estocástico (ej. CIR o Cox-Ingersoll-Ross) correlacionado con el precio del activo $S_t$:
$$d\lambda_t = \kappa(\theta - \lambda_t)dt + \sigma_\lambda \sqrt{\lambda_t} dW_t^\lambda$$
$$\text{Corr}(dW_t^S, dW_t^\lambda) = \rho$$
Si $\rho > 0$ (o negativo dependiendo de la dirección del trade), el CVA puede aumentar exponencialmente, no linealmente.

### 6.2 Sensibilidades del CVA (Las Griegas del CVA)
El CVA hace que el portafolio sea sensible a nuevos factores de riesgo. La mesa de operaciones debe gestionar:
*   **Credit Spread Delta:** Sensibilidad del CVA al cambio en los spreads de crédito de la contraparte. Se cubre comprando protección (CDS) sobre la contraparte.
*   **Cross-Gamma:** El CVA introduce convexidad cruzada entre el activo de mercado y el crédito. Si el mercado se mueve a favor del banco (aumenta exposición), y simultáneamente el crédito de la contraparte empeora, la pérdida de CVA se acelera. Esta es una de las sensibilidades más difíciles de cubrir en la práctica.

### 6.3 Implicaciones Regulatorias: Basilea III y el Capital
El cálculo del CVA no es solo un tema de valoración justa (*Fair Value* contable bajo IFRS 13), sino de solvencia regulatoria. Basilea III introdujo un cargo de capital por riesgo de CVA (*CVA Capital Charge*) para cubrir la **volatilidad** del CVA, no solo el riesgo de default. Esto se debió a que, durante la crisis de 2008, dos tercios de las pérdidas por riesgo de crédito de contraparte no se debieron a defaults reales, sino al deterioro de la calidad crediticia (aumento de spreads) que infló el CVA y generó pérdidas contables masivas en los libros de los bancos.

---

## 7. Conclusión

El cálculo del CVA representa la intersección moderna entre la ciencia actuarial, las matemáticas financieras y la tecnología computacional. Hemos demostrado que la exposición no es un número estático, sino un proceso estocástico complejo ($EE(t)$).

Para el actuario de riesgos, las conclusiones clave son:
1.  **La exposición es una distribución, no un punto:** El uso de promedios (EE) es válido para *pricing*, pero el uso de colas (PFE) es obligatorio para la gestión de capital y límites.
2.  **La computación es intensiva:** La necesidad de simular miles de escenarios y valorar instrumentos en cada nodo futuro exige el uso de técnicas de vectorización (mostradas en el código) o aceleración por GPU para carteras reales.
3.  **El riesgo es dinámico:** El CVA fluctúa con el mercado. Una posición rentable (In-the-money) conlleva implícitamente una posición larga en el riesgo de crédito de la contraparte que debe ser monitoreada y cubierta activamente.

Este marco de trabajo proporciona la base técnica para abordar derivados más complejos (Swaps, Bermudas) y ajustes adicionales como DVA (Debt Valuation Adjustment) y FVA (Funding Valuation Adjustment), completando así la familia de los XVA.

